# Shared Train/Validation/Test Split for Predictive Models

In [1]:
from datetime import datetime

import polars as pl

from run_config import (
    PATHS,
    RUN_MODE,
    MODEL_START_DATE,
    MODEL_END_DATE,
)

## Train Test Split

The input and output paths are selected by `RUN_MODE` in `run_config.py`. Thus, both `sample` and `full` mode write to separate locations. The generated 70/15/15 train, validation and test files are shared inputs for all predictive models. The random split is grouped by calendar date, so all spatial units and time buckets of a day remain in the same split. The checks below verify disjoint dates and report complete-day counts plus zero- and positive-demand coverage for every partition.

In [2]:
DATASETS = (
    PATHS.gold_1h_demand_hexagon,
    PATHS.gold_1h_demand_census_tracts,
    PATHS.gold_1h_demand_community_areas,
    PATHS.gold_1h_demand_community_area_unfiltered,
    PATHS.gold_2h_demand_hexagon,
    PATHS.gold_2h_demand_census_tracts,
    PATHS.gold_2h_demand_community_areas,
    PATHS.gold_2h_demand_community_area_unfiltered,
    PATHS.gold_4h_demand_hexagon,
    PATHS.gold_4h_demand_census_tracts,
    PATHS.gold_4h_demand_community_areas,
    PATHS.gold_4h_demand_community_area_unfiltered,
)
OUTPUT_DIR = PATHS.train_test_dir
TARGET_COL = "trip_count"

SEED = 42
RANDOM = True

MODEL_START_TS = datetime.fromisoformat(MODEL_START_DATE)
MODEL_END_TS = datetime.fromisoformat(MODEL_END_DATE)
if MODEL_START_TS >= MODEL_END_TS:
    raise ValueError(
        f"MODEL_START_DATE must be before MODEL_END_DATE: "
        f"{MODEL_START_DATE} >= {MODEL_END_DATE}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run mode: {RUN_MODE}")
if RUN_MODE == "full":
    print(f"Model period: [{MODEL_START_DATE}, {MODEL_END_DATE})")
else:
    print("Model-period filter disabled in sample mode")
print(f"Inputs: {[path.name for path in DATASETS]}")
print(f"Output directory: {OUTPUT_DIR}")

Run mode: full
Model period: [2025-01-01T00:00:00, 2026-05-01T00:00:00)
Inputs: ['GOLD_1H_DEMAND_HEXAGON_7.parquet', 'GOLD_1H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_2H_DEMAND_HEXAGON_7.parquet', 'GOLD_2H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_2H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_4H_DEMAND_HEXAGON_7.parquet', 'GOLD_4H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS.parquet']
Output directory: /Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data


In [3]:
def create_splits(dataset_path):
    df_split = pl.scan_parquet(dataset_path)

    if RUN_MODE == "full":
        df_split = df_split.filter(
            (pl.col("datetime_hour") >= MODEL_START_TS)
            & (pl.col("datetime_hour") < MODEL_END_TS)
        )

    if RANDOM:
        # Keep every spatial unit and time bucket from the same calendar day
        # in one split. Unique dates are ordered reproducibly by their hash.
        # Validation and test receive exactly the same number of complete days.
        date_assignment = (
            df_split
            .select(pl.col("datetime_hour").dt.date().alias("_split_date"))
            .unique()
            .with_columns(
                pl.col("_split_date").hash(seed=SEED).alias("_split_order")
            )
            .sort(["_split_order", "_split_date"])
            .collect()
            .with_row_index("_date_rank")
        )
        n_dates = date_assignment.height
        n_holdout_dates = round(n_dates * 0.15)
        n_train_dates = n_dates - 2 * n_holdout_dates
        if n_train_dates <= 0 or n_holdout_dates <= 0:
            raise ValueError(f"Not enough dates for grouped 70/15/15 split: {n_dates}")

        date_assignment = (
            date_assignment
            .with_columns(
                pl.when(pl.col("_date_rank") < n_train_dates)
                .then(pl.lit("train"))
                .when(pl.col("_date_rank") < n_train_dates + n_holdout_dates)
                .then(pl.lit("val"))
                .otherwise(pl.lit("test"))
                .alias("_split")
            )
            .select(["_split_date", "_split"])
        )
        bucketed = (
            df_split
            .with_columns(
                pl.col("datetime_hour").dt.date().alias("_split_date")
            )
            .join(date_assignment.lazy(), on="_split_date", how="inner")
        )
        train = bucketed.filter(pl.col("_split") == "train")
        val = bucketed.filter(pl.col("_split") == "val")
        test = bucketed.filter(pl.col("_split") == "test")
        helper_columns = ["_split_date", "_split"]
        train = train.drop(helper_columns)
        val = val.drop(helper_columns)
        test = test.drop(helper_columns)
    else:
        train = df_split.filter(pl.col("datetime_hour") < pl.datetime(2025, 9, 1))
        val = df_split.filter(
            (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1))
            & (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
        )
        test = df_split.filter(pl.col("datetime_hour") >= pl.datetime(2026, 1, 1))

    return df_split, train, val, test


split_results = {}
for dataset_path in DATASETS:
    df_split, train, val, test = create_splits(dataset_path)
    output_paths = {
        "train": OUTPUT_DIR / f"{dataset_path.stem}_TRAIN.parquet",
        "val": OUTPUT_DIR / f"{dataset_path.stem}_VAL.parquet",
        "test": OUTPUT_DIR / f"{dataset_path.stem}_TEST.parquet",
    }

    counts = {
        "total": df_split.select(pl.len()).collect().item(),
        "train": train.select(pl.len()).collect().item(),
        "val": val.select(pl.len()).collect().item(),
        "test": test.select(pl.len()).collect().item(),
    }
    if counts["total"] == 0:
        raise ValueError(f"Input dataset is empty: {dataset_path}")
    if counts["train"] + counts["val"] + counts["test"] != counts["total"]:
        raise ValueError(f"Split counts do not add up for {dataset_path}")

    split_dates = {
        name: frame.select(
            pl.col("datetime_hour").dt.date().alias("date")
        ).unique().collect()
        for name, frame in {"train": train, "val": val, "test": test}.items()
    }
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = split_dates[left].join(split_dates[right], on="date", how="inner")
        if overlap.height:
            raise ValueError(f"Date leakage between {left} and {right}: {overlap.height} dates")
    date_counts = {name: dates.height for name, dates in split_dates.items()}

    target_stats = {}
    for name, frame in {"train": train, "val": val, "test": test}.items():
        stats = frame.select(
            pl.col(TARGET_COL).is_null().sum().alias("null_targets"),
            (pl.col(TARGET_COL) == 0).sum().alias("zero_demand"),
            (pl.col(TARGET_COL) > 0).sum().alias("positive_demand"),
            pl.col(TARGET_COL).min().alias("min_demand"),
            pl.col(TARGET_COL).mean().alias("mean_demand"),
            pl.col(TARGET_COL).max().alias("max_demand"),
        ).collect().row(0, named=True)
        if stats["null_targets"]:
            raise ValueError(
                f"{dataset_path.name} {name} contains null target values"
            )
        if stats["min_demand"] < 0:
            raise ValueError(
                f"{dataset_path.name} {name} contains negative demand"
            )
        if stats["zero_demand"] == 0 or stats["positive_demand"] == 0:
            raise ValueError(
                f"{dataset_path.name} {name} must contain both zero- and "
                "positive-demand observations"
            )
        target_stats[name] = stats

    train.sink_parquet(output_paths["train"])
    val.sink_parquet(output_paths["val"])
    test.sink_parquet(output_paths["test"])
    split_results[dataset_path.stem] = {
        "counts": counts,
        "date_counts": date_counts,
        "target_stats": target_stats,
        "paths": output_paths,
    }

    shares = {name: round(count / counts["total"], 2) for name, count in counts.items() if name != "total"}
    print(f"{dataset_path.name}: {counts}, shares={shares}")
    print(f"Grouped split dates: {date_counts}")
    print(f"Target coverage: {target_stats}")
    print(f"Written: {output_paths}")

GOLD_1H_DEMAND_HEXAGON_7.parquet: {'total': 1408440, 'train': 984456, 'val': 211992, 'test': 211992}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Written: {'train': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_TRAIN.parquet'), 'val': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_VAL.parquet'), 'test': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_TEST.parquet')}
GOLD_1H_DEMAND_CENSUS_TRACTS.parquet: {'total': 10219920, 'train': 7143408, 'val': 1538256, 'test': 1538256}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Written: {'train': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_CENSUS_TRACTS_TRAIN.parquet'), 'val': PosixPath('/Users/lennartjekel/dev/

In [4]:
df_split.head(10).collect()

datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,tmpc,relh,sknt,vsby,p01m,skyc1_BKN,skyc1_CLR,skyc1_FEW,skyc1_OVC,skyc1_SCT,skyc1_VV,date,is_holiday,community_area,food_drink,landmark,shop,train_station,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,date,i8,i64,f64,f64,f64,f64,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
2025-02-03 16:00:00,2,1,16,0.5,0.866025,0.0,1.0,-0.866025,-0.5,6.268571,81.27,11.0,3.5,0.0,0,0,0,1,0,0,2025-02-03,0,11,38.0,10.0,13.0,4.0,1,2280,2280.0,2280,2280,10.7,10.7,10.7,10.7,30.0,30.0,30.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,30.0,30.0,30.0,"""Unknown"""
2025-01-20 04:00:00,1,1,4,0.0,1.0,0.0,1.0,0.866025,0.5,-16.6675,49.13,11.25,10.0,0.0,0,1,0,0,0,0,2025-01-20,1,11,38.0,10.0,13.0,4.0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2025-01-28 08:00:00,1,2,8,0.0,1.0,0.781831,0.62349,0.866025,-0.5,-0.2775,57.0225,6.25,10.0,0.0,0,1,0,0,0,0,2025-01-28,0,11,38.0,10.0,13.0,4.0,5,5505,1101.0,228,2668,30.33,6.066,0.85,11.15,93.75,18.75,7.5,30.75,2.81,0.562,0.0,2.81,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97.06,19.412,7.5,30.75,"""Prcard"""
2025-02-04 04:00:00,2,2,4,0.5,0.866025,0.781831,0.62349,0.866025,0.5,1.35,73.202857,11.285714,10.0,0.0,0,0,0,1,0,0,2025-02-04,0,11,38.0,10.0,13.0,4.0,1,1743,1743.0,1743,1743,12.0,12.0,12.0,12.0,30.75,30.75,30.75,30.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.75,30.75,30.75,30.75,"""Prcard"""
2025-02-19 20:00:00,2,3,20,0.5,0.866025,0.974928,-0.222521,-0.866025,0.5,-9.165,45.74,10.5,10.0,0.0,0,0,1,0,0,0,2025-02-19,0,11,38.0,10.0,13.0,4.0,2,3034,1517.0,785,2249,33.91,16.955,12.08,21.83,84.69,42.345,30.75,53.94,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,84.69,42.345,30.75,53.94,"""Prcard"""
2025-01-23 00:00:00,1,4,0,0.0,1.0,0.433884,-0.900969,0.0,1.0,-4.178,64.136,10.9,6.125,0.001,1,0,0,0,0,0,2025-01-23,0,11,38.0,10.0,13.0,4.0,1,1080,1080.0,1080,1080,5.6,5.6,5.6,5.6,16.75,16.75,16.75,16.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16.75,16.75,16.75,16.75,"""Cash"""
2025-02-02 08:00:00,2,7,8,0.5,0.866025,-0.781831,0.62349,0.866025,-0.5,1.111429,69.308571,9.857143,9.714286,0.0004,0,0,0,1,0,0,2025-02-02,0,11,38.0,10.0,13.0,4.0,1,3300,3300.0,3300,3300,7.9,7.9,7.9,7.9,35.0,35.0,35.0,35.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.0,35.0,35.0,35.0,"""Cash"""
2025-02-21 16:00:00,2,5,16,0.5,0.866025,-0.433884,-0.900969,-0.866025,-0.5,-2.78,51.535,10.0,10.0,0.0,0,1,0,0,0,0,2025-02-21,0,11,38.0,10.0,13.0,4.0,2,3300,1650.0,960,2340,13.2,6.6,5.5,7.7,40.0,20.0,16.25,23.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,20.0,16.25,23.75,"""Cash"""
2025-02-06 04:00:00,2,4,4,0.5,0.866025,0.433884,-0.900969,0.866025,0.5,-0.9725,88.48,4.5,2.5,0.0004,0,0,0,1,0,0,2025-02-06,0,11,38.0,10.0,13.0,4.0,2,3383,1691.5,896,2487,14.59,7.295,2.54,12.05,43.5,21.75,11.75,31.75,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.5,0.0,1.0,44.5,22.25,12.75,31.75,"""Prcard"""
